In [1]:
import sqlite3
from pathlib import Path

db_path = Path(r"C:\Users\Franc\OneDrive\Documents\GitHub\test_arkose\database\climbing.db")
conn = sqlite3.connect(db_path, check_same_thread=False)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('boulders',), ('ascents',), ('sqlite_sequence',), ('sync_log',), ('comments',)]


In [ ]:
cursor.execute("")

In [3]:
cursor.execute("PRAGMA table_info(boulders);")

for column in cursor.fetchall():
    print(column)

(0, 'boulder_id', 'TEXT', 0, None, 1)
(1, 'gym', 'TEXT', 1, None, 0)
(2, 'grade', 'TEXT', 0, None, 0)
(3, 'label', 'INTEGER', 0, None, 0)
(4, 'boulder_num', 'INTEGER', 0, None, 0)
(5, 'zone', 'INTEGER', 0, None, 0)
(6, 'holds_color', 'INTEGER', 0, None, 0)
(7, 'route_setter', 'TEXT', 0, None, 0)
(8, 'route_types', 'TEXT', 0, None, 0)
(9, 'created_at', 'TEXT', 0, None, 0)
(10, 'closed_at', 'TEXT', 0, None, 0)
(11, 'sents_count', 'INTEGER', 0, '0', 0)
(12, 'flashes_count', 'INTEGER', 0, '0', 0)
(13, 'first_seen_at', 'TEXT', 1, None, 0)
(14, 'last_updated_at', 'TEXT', 1, None, 0)


In [4]:
cursor.execute("PRAGMA table_info(ascents);")

for column in cursor.fetchall():
    print(column)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'boulder_id', 'TEXT', 1, None, 0)
(2, 'user_id', 'TEXT', 1, None, 0)
(3, 'ascent_type', 'TEXT', 1, None, 0)
(4, 'detected_at', 'TEXT', 1, None, 0)


In [6]:
from climbing_coach.src.climber_profile import ClimberProfile
DB_PATH = r"C:\Users\Franc\OneDrive\Documents\GitHub\test_arkose\database\climbing.db"  # remplace par ton Path absolu
GYM = "arkose/montmartre"
USER_ID = "qQFsxQKYvqRqYJNKa"

profile = ClimberProfile.load(r"C:\Users\Franc\OneDrive\Documents\GitHub\test_arkose\database\franck.json")

In [9]:
import sqlite3
from climbing_coach.src.climbing_coach import StatsBuilder  # adapte le nom de fichier si besoin
from climbing_coach.src.sboulder_collector import decode_grade, sboulder_url, _now_iso
from climbing_coach.src.sboulder_collector import encode_grade_level

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

# --- 1. Inspecter une voie précise par son boulder_id ---
def inspect_boulder(boulder_id):
    print(f"\n=== Inspection de {boulder_id} ===")
    row = conn.execute("SELECT * FROM boulders WHERE boulder_id = ?", (boulder_id,)).fetchone()
    if not row:
        print("❌ Boulder introuvable en base !")
        return
    d = dict(row)
    print("Données brutes :", d)
    print("Grade décodé   :", decode_grade(d["holds_color"], d["grade"]))
    print("URL générée    :", sboulder_url(d["gym"], boulder_id))
    print("Fermé ?        :", d["closed_at"] is not None and d["closed_at"] <= _now_iso())

    comments = conn.execute(
        "SELECT * FROM comments WHERE boulder_id = ?", (boulder_id,)
    ).fetchall()
    print(f"Commentaires en base ({len(comments)}) :")
    for c in comments:
        print("  -", dict(c))

# --- 2. Lister toutes les voies "noir 3 barres" ouvertes, pour comparer ---
print("=== Toutes les voies NOIR 3 barres ouvertes à Montmartre ===")
now = _now_iso()
rows = conn.execute(
    "SELECT boulder_id, grade, holds_color, closed_at, sents_count "
    "FROM boulders WHERE gym = ? AND holds_color = 7 AND grade = '3' "
    "AND (closed_at IS NULL OR closed_at > ?)",
    (GYM, now)
).fetchall()
for r in rows:
    print(dict(r), "| url:", sboulder_url(GYM, r["boulder_id"]))

# --- 3. Comparer avec ce que StatsBuilder envoie réellement au LLM ---
print("\n=== Ce que StatsBuilder/to_llm_context envoie réellement ===")
sb = StatsBuilder(DB_PATH)
stats = sb.build(user_id=USER_ID, gyms=[GYM], min_level=encode_grade_level(profile.current_flash_level_arkose))
print(stats.to_llm_context())

inspect_boulder("fzapQZzDCNwcAQHDs")

conn.close()

=== Toutes les voies NOIR 3 barres ouvertes à Montmartre ===
{'boulder_id': 'HtuQgH6cma7ZwEYuL', 'grade': '3', 'holds_color': 7, 'closed_at': '2026-08-24T21:00:49.266000+00:00', 'sents_count': 27} | url: https://www.sboulder.com/arkose/montmartre?b=HtuQgH6cma7ZwEYuL
{'boulder_id': 'LP7KXwACFYhyogywu', 'grade': '3', 'holds_color': 7, 'closed_at': '2026-08-14T22:00:00+00:00', 'sents_count': 38} | url: https://www.sboulder.com/arkose/montmartre?b=LP7KXwACFYhyogywu
{'boulder_id': 'fzapQZzDCNwcAQHDs', 'grade': '3', 'holds_color': 7, 'closed_at': '2026-08-14T22:00:00+00:00', 'sents_count': 58} | url: https://www.sboulder.com/arkose/montmartre?b=fzapQZzDCNwcAQHDs
{'boulder_id': '2zbeebXqiNn4nZn8f', 'grade': '3', 'holds_color': 7, 'closed_at': '2026-08-10T16:24:52.585000+00:00', 'sents_count': 121} | url: https://www.sboulder.com/arkose/montmartre?b=2zbeebXqiNn4nZn8f
{'boulder_id': 'dNNBHeTnNkmTK3HeZ', 'grade': '3', 'holds_color': 7, 'closed_at': '2026-08-09T16:00:21.424000+00:00', 'sents_coun


=== Inspection de fzapQZzDCNwcAQHDs ===


ProgrammingError: Cannot operate on a closed database.